In [ ]:



from pathlib import Path
import json,mlflow,numpy as np,pandas as pd
#remember artifacts are stored in mlflow.db, so you need to set the tracking uri to the sqlite db
#they are any file output logged alongside a run, such as a model, image, or data file. Artifacts are stored in the artifact store, which is a location where MLflow can save and retrieve files. The artifact store can be a local file system, an Amazon S3 bucket, an Azure Blob Storage container, or a Google Cloud Storage bucket.
from mlflow.tracking import MlflowClient 
from src.modeling.constraints import check,load_artifacts

from src.modeling.constraints import setup_mlflow
setup_mlflow()



client = MlflowClient()

run_id = "25bd827ce16d409480c09b0dd37626ea"

a=load_artifacts(run_id,client)

model=mlflow.sklearn.load_model("models:/wine_quality_rf/1")


print(model.n_features_in_, a["mahal"]["threshold_d2"])


In [ ]:
print(mlflow.get_tracking_uri())

In [ ]:
#assigning the roles 
#in the mahal i saved the feauture order 
feature_order=a["mahal"]["feature_order"]

ROLES = {
    "lever":   ["free sulfur dioxide", "sulphates", "citric acid"],
    "semi":    ["fixed acidity", "pH", "residual sugar"],
    "outcome": ["total sulfur dioxide", "density", "volatile acidity", "alcohol"],
    "context": ["chlorides"],
}


all_roles=[f for v in ROLES.values() for f in v]
assert sorted(all_roles)==sorted(feature_order)

LEVERS=ROLES["lever"]
LEVER_IDX=[feature_order.index(f) for f in LEVERS]

#we will use nonlinearconstraint because we have
# total SO₂ ≤ 200 mg/L if residual sugar < 5 g/L
#total SO₂ ≤ 250 mg/L αν residual sugar ≥ 5 g/L






In [ ]:
print(a["bounds"])

In [ ]:
print(a["legal"])

In [ ]:
#lets assign the layers back
L1=a["bounds"]["bounds"]
L0=a["legal"]

BOUNDS=[]
for f in feature_order:
    lo,hi=L1[f]
    #if f has a legal cap ->hi=min
    assert lo<hi, f"empty interval for {f}"
    BOUNDS.append((lo,hi))


for f,b in zip(feature_order,BOUNDS):
    print(f,b)

our data are more strict that the law

In [ ]:
#we are going to use hi-lo for scaling 

SCALE=np.array([hi-lo for lo,hi in BOUNDS])

assert (SCALE > 0).all(), "zero or negative scale"




In [ ]:
#lets search for the experiment 

EXP_ID="1"

runs=client.search_runs(
    experiment_ids=[EXP_ID],
    filter_string="tags.role='phase6_source_of_truth' "
)

for r in runs:
    print(r.info.run_id,r.data.tags.get("mlflow.runName"))

    



In [ ]:

#to be sure
assert len(runs)==1, f"expected 1 run, got {len(runs)}"

phase4_run_id=runs[0].info.run_id

phase4_run_id

for f in client.list_artifacts(phase4_run_id):
    print(f.path ,f.is_dir)

In [ ]:
print(phase4_run_id)
print(phase4_run_id == "25bd827ce16d409480c09b0dd37626ea")

In [ ]:
#lets load the split 

split=mlflow.artifacts.load_dict(f"runs:/{phase4_run_id}/split_indices.json")


print(len(split["train_idx"]),len(split["test_idx"]))

In [ ]:
#load again the data

df=pd.read_csv("/home/nasia/wine-innovation-engine/data/raw/winequality-white (1).csv", sep=";")

X=df[feature_order]#we want them to be in the same order,the feauture orde is a list of names without the quality
y=df["quality"]

#lets split them 
X_test=X.loc[split["test_idx"]]#from split dictionary thake the test_idx
y_test=y.loc[split["test_idx"]]

print(X_test.shape)

In [ ]:
#finding the low quality

low_idx=y_test[y_test<=5].index

print(len(low_idx))
#getting the index of first with low quality 
idx=low_idx[0]

x0=X.loc[idx,feature_order].to_numpy(dtype=float)

x0



In [ ]:
#because we will call the check many times lets frooze everything excepts x 
#it is like that  check(x0, a["bounds"], a["mahal"], a["density_band"], a["legal"])

from functools import partial 
#partial makes a new function with the a lot of the things stable and filled 

feasible=partial(check,
                bounds=L1,
                mahal=a["mahal"],
                density_band=a["density_band"],
                legal=a["legal"]) 




In [ ]:
print("index      :", idx)
print("true quality:", y_test.loc[idx])
print("predicted   :", model.predict(x0.reshape(1,-1))[0])



#lets make panda series and numpy arrays
x0_s=X.loc[idx,feature_order]
x0=x0_s.to_numpy(dtype=float)

x0_s


feasible(x0_s) #the check wants panda series 

In [ ]:
#lets see again some bounds and some thresholds

for f,b in zip(feature_order,BOUNDS):
    print(f,b)



print("d^2 threshold:", a["mahal"]["threshold_d2"])



the cells output 

index      : 3578
true quality: 4
predicted   : 4.392887029887029



'L1_bounds': False,
 
 
 
'L2_mahal': True,

'L3_density': True,

'L0_legal': True,

'd2': 4.824290645618511,

'z_density': 0.46006072596582764} #z_density asks how many standard deviations is the actual density away from that predicted by the natural law density=f(alcohol, sugar)
 

so the free sulfur dioxide is 5 and is lower than the lower bound (5,67)  and also the we found is 4,82 and the threshold is 10.15 that means that the wine is inside the mahalanobis and also we need to keep in mind that the L2 asks if the combination is typical.

the 5 mg/l free SO2 in a white wine is very low ->the wine is essentially unprotected from oxidation. And SHAP had shown you free SO₂ as the #2 factor with a saturation threshold of ~30 mg/L.

In [ ]:
#we need a second x0 that can be true in the four layers because differentail evolution needs to start from a feasible point

cand=[i for i in y_test[y_test==5].index
      if all(v for k,v in feasible(X.loc[i,feature_order]).items()#feasible returns a dictionary that why we want the items
             if isinstance(v,bool))]


print(len(cand))

In [ ]:
#lets keep one good

idx_good=cand[0]
x0ob_s=X.loc[idx_good,feature_order]
x0b=x0ob_s.to_numpy(dtype=float)

print("index  :", idx_good)
print("true quality:",y_test.loc[idx_good])
print("predicted   :", model.predict(x0b.reshape(1,-1))[0])

print("feasible:",feasible(x0ob_s))

In [ ]:
#lets use the x0b (its a feasible wine)

#lets make a objective that calls another 
#it takes model->the random forest
#x0 the initial wine
#SCALE ->for every feature the price 
#lam how heavy is the 'switching cost'
def make_objective(model,x0,SCALE,lam):
    x0=np.asarray(x0,dtype=float)

    def objective(x):#x0 is the initial wine and x is the candidate 
        q=model.predict((x.reshape(1,-1)))[0]
        cost=np.sum(np.abs(x-x0)/SCALE)
        return -q+lam*cost #high quality and short distance ,DE ALWAYS MINIMIZE THATS WHY WE PUT MINIMIZES -Q   
#Small lam → the optimizer doesn't care how much it changes, it only chases quality.

    return objective

In [ ]:
x0b

In [ ]:
#sanity check lets run it in the x0b

obj=make_objective(model,x0b,SCALE,lam=0.1)

print(obj(x0b))

print(-model.predict(x0b.reshape(1,-1))[0])
#we expect zero because we compare it with his self ,so obj(x0b) must be the same with -model.predict(x0b)

In [ ]:
#mahalanobis distance

print(a["mahal"].keys())

In [ ]:
def mahalanobis_d2(x,mean,precision):
    diff=x-mean
    return diff@precision@diff    #it gives a dot product the d^2->d² is the squared Mahalanobis distance ,our L2 layer's measure of "how weird is this wine."

we are going to use NONLINEAR CONSTRAINTS(we have nonlinear bounds) -> They make optimization significantly harder
Curved boundaries can create non-convex search spaces
meaning your algorithm might get stuck in a "local valley" (local minimum) and miss the absolute best solution (global minimum).

In [ ]:
#NonlinearConstraint 

from scipy.optimize import NonlinearConstraint
#location is the mean
mean = np.array(a["mahal"]["location_"])
precision=np.array(a["mahal"]["precision_"])

def mahal_constraint_fn(x):
    return mahalanobis_d2(x,mean,precision)


nlc_mahal=NonlinearConstraint(mahal_constraint_fn,lb=-np.inf,ub=a["mahal"]["threshold_d2"]) #we care about the upper limit so that why we put -np.inf 

#lb ->lower bound
#the constraint function calculates d2=5.5
#nonlinearconstraint ->asks is 5.5 inside [-inf,10.15]


In [ ]:
#sanity check 
print(mahal_constraint_fn(x0b),"vs threshold",a["mahal"]["threshold_d2"])

In [ ]:
#lets see the density band 
print(a["density_band"].keys())

In [ ]:
print(a["density_band"])

target->which feature you predict->density
predictors-> the feautures you use 


In [ ]:
print(a["density_band"]["predictors"])


In [ ]:
#reminder k=the number of predictors
#sigma ->residual standard error 
#find the index in the feature order
i_dens=feature_order.index("density")

i_alc   = feature_order.index("alcohol")
i_alc
i_sugar = feature_order.index("residual sugar")
i_dens,i_alc,i_sugar

In [ ]:

c = a["density_band"]["coef"]
sigma = a["density_band"]["sigma"]
k = a["density_band"]["k"]

print(c)


print(sigma)
print(k)

In [ ]:
def density_z_fn(x):#regression equation we created in Phase 5 density = 1.00509 + (-0.00126)·alcohol + 0.000350·residual_sugar
    predicted=c["const"]+c["alcohol"]*x[i_alc]+c["residual sugar"]*x[i_sugar]

    return abs(x[i_dens]-predicted)/sigma #->the difference between the density claimed by the candidate wine and that imposed by physics. This is called residual
#we divide with sigma so we can read it from the perspective 
#we write it a way DE understands it 


#For each candidate wine: calculate how many standard deviations its density is away from that imposed by the law density = f(alcohol, sugar), and reject it if it exceeds 3.'
nlc_density=NonlinearConstraint(density_z_fn,lb=-np.inf,ub=k) #k=3.0-> 'Do not exceed 3 standard deviations', lb=-np.inf (infinive)

In [ ]:
#sanity check
print(density_z_fn(x0b)," k=",k)

#the L0 is the reason we choose nonlinear constraints 
# Here the limit of total SO₂ moves according to the residual sugar

total SO₂ ≤ 200 mg/L   if residual sugar < 5 g/L




total SO₂ ≤ 250 mg/L   if residual sugar ≥ 5 g/L

In [ ]:
i_tso2  = feature_order.index("total sulfur dioxide")
i_sugar = feature_order.index("residual sugar")

so2 = L0["total sulfur dioxide"]





In [ ]:
def so2_margin_fn(x):
    cap=so2["cap_high_sugar"] if x[i_sugar]>=so2["sugar_threshold_gL"] else so2["cap_low_sugar"]
    return cap-x[i_tso2] #it returns περιθώριο (cap − actual)
#if it is positive i can go further if its not icant 

nlc_so2 = NonlinearConstraint(so2_margin_fn, lb=0, ub=np.inf)

#sanity check
print("x0b:", so2_margin_fn(x0b), " sugar:", x0b[i_sugar], " tSO2:", x0b[i_tso2])

 Wine has 79 mg/L margin before reaching the legal limit.

In [ ]:
#we only changed the sugar, left the tSO₂ at 171
xt = x0b.copy(); xt[i_sugar] = 2.0;  print("dry :", so2_margin_fn(xt))
xt = x0b.copy(); xt[i_sugar] = 10.0; print("sweet:", so2_margin_fn(xt))

so If we want to raise total SO₂ above 200 (because SHAP says SO₂ helps), we has two options:

stay below 200 and lose the benefit
keep sugar above 5g/L and unlock 250

In [ ]:
#teaching the DE the free SO₂ ≤ total SO₂

i_fso2=feature_order.index("free sulfur dioxide")

def free_le_total_fn(x):
    return x[i_tso2]-x[i_fso2]


nlc_free=NonlinearConstraint(free_le_total_fn,lb=0,ub=np.inf)



In [ ]:
#sanity check
print("margin:",free_le_total_fn(x0b),
      " free:", x0b[i_fso2], " total:", x0b[i_tso2])

In [ ]:
#lets see example to see if this breaks

xt=x0b.copy();xt[i_fso2]=200
print("violation:",free_le_total_fn(xt))#if its negative it works

In [ ]:
#lets pick all of the constraints 
CONSTRAINTS = [nlc_mahal, nlc_density, nlc_so2, nlc_free]

from scipy.optimize import differential_evolution

obj=make_objective(model,x0b,SCALE,lam=1.5)


res=differential_evolution(
    obj,
    bounds=BOUNDS,
    constraints=CONSTRAINTS,
    seed=42,
    polish=False,
    maxiter=200,
    popsize=30,
    init='latinhypercube'
)

print(res.success, res.message)
print("objective:", res.fun)

In [ ]:
#the objective was -q+λ*cost and we have as a result -7.33
#but we need to separate them to understand what is the cost and what the quality


q_new=model.predict(res.x.reshape(1,-1))[0]
cost_new=np.sum(np.abs(res.x-x0b)/SCALE)

LAMBDA=1.5
print("quality:", model.predict(x0b.reshape(1,-1))[0], "→", q_new)
print("cost   :", cost_new, " (penalty =", LAMBDA*cost_new, ")") #λ=0.3
print("check  : -q + λ·cost =", -q_new + LAMBDA*cost_new)


#LETS keep SOME OF THE VARIABLES
q0     = float(model.predict(x0b.reshape(1, -1))[0])   # baseline quality
q_star = float(model.predict(res.x.reshape(1, -1))[0]) # final quality
cost   = float(cost_new)                   

In [ ]:
#making a delta table to translate the results 

delta=pd.DataFrame({
    "before":x0b,
    "after":res.x,
    "delta":res.x-x0b,
    "pct_range":(res.x-x0b)/SCALE,#the difference but normalised
},index=feature_order).round(4)

print(delta.sort_values("pct_range", key=abs, ascending=False))



before     after   delta  pct_range
alcohol                 9.1000   12.2529  3.1529     0.6708
density                 0.9987    0.9974 -0.0013    -0.1128
chlorides               0.0470    0.0393 -0.0077    -0.0514
pH                      3.0900    3.1263  0.0363     0.0511
total sulfur dioxide  171.0000  168.3772 -2.6228    -0.0136
residual sugar         15.3000   15.0896 -0.2104    -0.0118
citric acid             0.5100    0.5132  0.0032     0.0046
sulphates               0.5100    0.5078 -0.0022    -0.0041
free sulfur dioxide    54.0000   53.8447 -0.1553    -0.0021
fixed acidity           7.9000    7.8993 -0.0007    -0.0002
volatile acidity        0.3450    0.3450 -0.0000    -0.0001



ROLES = {
    "lever":   ["free sulfur dioxide", "sulphates", "citric acid"],
    "semi":    ["fixed acidity", "pH", "residual sugar"],
    "outcome": ["total sulfur dioxide", "density", "volatile acidity", "alcohol"],
    "context": ["chlorides"],
}


we see the biggest increasment is in the alcohol the rest are not that important,the model worked in a vintage-design way (all features free),it is optimal for a design space exploration.Alchohol is an outcome so we can change it in the middle only if we want the fermentation from the start with a specific percentage of alcohol.

In [ ]:

x_star = pd.Series(res.x, index=feature_order)
feasible(x_star)
x_star
audit = feasible(x_star)
{k: type(v) for k, v in audit.items()}

In [ ]:
#logging everything

from mlflow import MlflowClient 

client = MlflowClient(tracking_uri="sqlite:////home/nasia/wine-innovation-engine/notebooks/mlflow.db") 

exp=client.get_experiment_by_name('phase6_reformulation')
exp_id=exp.experiment_id if exp else client.create_experiment("phase6_reformulation")

WINE_IDX=536
x0b=df.iloc[WINE_IDX][feature_order].values
assert np.allclose(x0b, df.iloc[WINE_IDX][feature_order].values), \
    "WINE_IDX και x0b δεν συμφωνούν — δείχνουν διαφορετικά κρασιά"

run=client.create_run(exp_id,tags={"role": "phase6_optimization",
    "scenario": "vintage_design",      
    "wine_index": str(WINE_IDX),
})

rid=run.info.run_id
LAMBDA=1.5
for k, v in {
    "quality_baseline": q0, "quality_final": q_star,
    "quality_gain": q_star - q0, "proximity_cost": cost,
    "penalty": LAMBDA * cost, "objective_check": -q_star + LAMBDA * cost,
    "de_success": int(res.success), "de_nfev": res.nfev,
}.items():
    client.log_metric(rid, k, float(v))

client.log_dict(rid, delta.reset_index().to_dict(orient="list"), "delta_table.json")
client.log_dict(rid, dict(zip(feature_order, res.x.tolist())), "solution_vector.json")
client.log_dict(rid, feasible(x_star), "constraint_audit.json")   

client.set_terminated(rid)


"Implemented actionable algorithmic recourse over a Random Forest quality model, constrained by EU regulatory bounds and physicochemical feasibility layers."

We are implementing counterfactual (vintage design) and minimal intervention (cellar intervention) according to Algorithmic Recourse:
from Counterfactual Explanations to Interventions(Karimi et al,2020)

actual instance x_F	x0b ->our #536 
counterfactual x_CFE	->result  vintage_design — 
minimal intervention	->result cellar_intervention — actionable
feasibility set F	->4 constraint layers (L0–L3)
cost function	->λ · proximity_cost (L1 normalized)
actionable features	LEVERS-> free SO₂, sulphates, citric acid
immutable features->ROLES["outcome"]: alcohol, density

only winemaker-controllable variables (levers) are free, outcome variables are locked at baseline, approximating causal intervention without a formal SCM.



In [ ]:
#STRICT cellar intervetion ,we need to give a start 
#change only(the levers)the free sulfur dioxide,sulphates,citric acid
#we semi-levers,constrained out

LEVERS = {"free sulfur dioxide", "sulphates", "citric acid"}

BOUNDS_CELLAR=[]
for i, feat in enumerate(feature_order):#we use enumerate to take both the index and the name from the feature order to ask if it lever and take the bounds
    if feat in LEVERS:
        BOUNDS_CELLAR.append(BOUNDS[i])
    else:#IT TAKES FOR EXAMPLE THE ALCOHOL AND THE PRICE 
        v=float(x0b[i])#x0b is the start of the optimizer before starting to keep getting better, #BASELINE POINT
        BOUNDS_CELLAR.append((v,v)) #v-v takes the baseline price and it locks it ,

#JSON NEEDS FLOAT AND TOLIST TO OPERATE IT DOES NOT UNDERSTAND NUMPY

In [ ]:
print("vintage_design logged:", rid)
print("cellar_intervention cell status:", BOUNDS_CELLAR[:3])

In [ ]:
#lets run the optimizer only for the cellar_intervention

res_cellar=differential_evolution(
    obj,
    bounds=BOUNDS_CELLAR,
    constraints=CONSTRAINTS,
    seed=42,
    polish=False,
    maxiter=200,
    popsize=30,
    init='latinhypercube'
)

print(res_cellar.success, res_cellar.message) #the success  says if its okay with the constraints 
print("objective:", res_cellar.fun)

x_star_cellar = pd.Series(res_cellar.x, index=feature_order)
print(x_star_cellar[list(LEVERS)])

In [ ]:

delta=pd.DataFrame({
    "before":x0b,
    "after":x_star_cellar,
    "delta":x_star_cellar-x0b,
    "pct_range":(x_star_cellar-x0b)/SCALE,#the difference but normalised
},index=feature_order).round(4)

print(delta.sort_values("pct_range", key=abs, ascending=False))

In [ ]:
x_star_cellar = pd.Series(res_cellar.x, index=feature_order)
audit_cellar = feasible(x_star_cellar)

exp = client.get_experiment_by_name('phase6_reformulation')
exp_id = exp.experiment_id if exp else client.create_experiment("phase6_reformulation")

q0 = float(model.predict([x0b])[0])
q_star_cellar = float(model.predict([res_cellar.x])[0])
cost_cellar = float(np.sum(np.abs(res_cellar.x - x0b) / SCALE))

LAMBDA = 1.5

run_cellar = client.create_run(exp_id, tags={
    "role": "phase6_optimization",
    "scenario": "cellar_intervention",   # ← αλλαγή 1
    "wine_index": str(WINE_IDX),
})

rid_cellar = run_cellar.info.run_id

for k, v in {
    "quality_baseline": q0,
    "quality_final": q_star_cellar,
    "quality_gain": q_star_cellar - q0,
    "proximity_cost": cost_cellar,
    "penalty": LAMBDA * cost_cellar,
    "objective_check": -q_star_cellar + LAMBDA * cost_cellar,
    "de_success": int(res_cellar.success),
    "de_nfev": res_cellar.nfev,
}.items():
    client.log_metric(rid_cellar, k, float(v))

delta_cellar = pd.DataFrame({
    "before": x0b,
    "after": res_cellar.x,
    "delta": res_cellar.x - x0b,
    "pct_range": (res_cellar.x - x0b) / SCALE,
}, index=feature_order).round(4)

client.log_dict(rid_cellar, delta_cellar.reset_index().to_dict(orient="list"), "delta_table.json")
client.log_dict(rid_cellar, dict(zip(feature_order, res_cellar.x.tolist())), "solution_vector.json") 

client.set_terminated(rid_cellar)
print("logged:", rid_cellar)